In [1]:
#libraries
import numpy as np
import sympy as sp
import sympy.physics.mechanics as me
import matplotlib.pyplot as plt
from anastruct import SystemElements


In [2]:


def Pin_locator (theta_deg, L, pin_name=None):
   

    theta = np.radians(theta_deg)   # Convert degrees to radians
    x = L*np.cos(theta)
    y = L*np.sin(theta)
    L_h=L/2
    x_h = L_h*np.cos(theta)
    y_h = L_h*np.sin(theta)

    H = np.array([0, L]).astype(float)  
    A = H + np.array([0, -y]).astype(float)  
    P1 = H + np.array([x_h, -y_h]).astype(float)  
    B = A + np.array([x, y]).astype(float)  
    C = H + np.array([x, -y]).astype(float)  
    P2 = B + np.array([x_h, -y_h]).astype(float)  
    E = B + np.array([x, -y]).astype(float)  
    D = C + np.array([x, y]).astype(float)  
     
    
    # Dictionary of all pins
    pins_dict = {'H': H, 'A': A, 'P1': P1, 'B': B, 'C': C, 'P2': P2, 
                 'E': E, 'D': D}
    
    
    if pin_name is None:
        return H, A, P1, B, C, P2, E, D
    else:
        if pin_name not in pins_dict:
            raise ValueError(f"Pin '{pin_name}' not found. Available pins: {list(pins_dict.keys())}")
        
        return pins_dict[pin_name]
    


In [3]:

# --- Constants ---
E_al = 70e9        
rho_al = 2700     
g = 9.81

def run_doe_simulation(theta_deg, L, A_cs, I_val, Px_dist):
    """
    Returns the max tension and compression for a specific set of parameters.
    """
    # 1. Setup system and weights
    weight_per_meter = rho_al * A_cs * g
    ss = SystemElements(EA=float(E_al * A_cs), EI=float(E_al * I_val))
    
    # 2. Get Coordinates (Assuming Pin_locator is defined elsewhere in your script)
    pin_names = ['H', 'A', 'P1', 'B', 'C', 'P2', 'E', 'D']
    coords = {name: Pin_locator(theta_deg, L, pin_name=name) for name in pin_names}
    
    # 3. Connectivity
    elements = [
        ('H', 'P1'), ('P1', 'C'), ('A', 'P1'), ('P1', 'B'), 
        ('B', 'P2'), ('P2', 'E'), ('C', 'P2'), ('P2', 'D'), 
        ('E', 'D')                                         
    ]

    # 4. Build Structure
    for n1, n2 in elements:
        ss.add_element(location=[coords[n1], coords[n2]], g=float(weight_per_meter))
    
    # 5. Boundary Conditions
    id_H = ss.find_node_id(coords['H'])
    id_A = ss.find_node_id(coords['A'])
    ss.add_support_hinged(node_id=id_H)
    ss.add_support_hinged(node_id=id_A)

    # 6. Apply Distributed Load (on the top member E-D)
    # Finding the element ID for ('E', 'D') - it's the last one added (ID 9)
    ss.q_load(element_id=9, q=Px_dist)
    
    # 7. Solve
    ss.solve()
# --- 8. Extract Results (Corrected Logic) ---
    axial_forces = np.array(ss.get_element_result_range('axial'))
    
    # Tension: Max positive value (if none, then 0)
    raw_max = np.max(axial_forces)
    peak_force = raw_max if raw_max > 0 else 0
    

    
    return peak_force

In [5]:
test_force = run_doe_simulation(25,0.5,0.0050,1e-5,-4.59)
print(f"Test Force: {test_force:.4f} N")

Test Force: 685.9852 N
